In [1]:
#2
import sys
from pathlib import Path

# Permite importar lista4 desde la carpeta src
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from lista4.data import load_csv
from lista4.models import linear_summary
from lista4.metrics import corr_pair

archivos = ["experimento_2_a", "experimento_2_b", "experimento_2_c"]

In [2]:
#A
correlaciones = {}
for nombre in archivos:
    d = load_csv(nombre)
    correlaciones[nombre] = corr_pair(d, "x1", "x2")

pd.Series(correlaciones, name="corr(x1, x2)").to_frame()

,"corr(x1, x2)"
experimento_2_a,0.999760
experimento_2_b,0.999779
experimento_2_c,0.999821


### (a) Correlación

| Archivo | corr(x1, x2) |
|---|---|
| experimento_2_a | 0.999760 |
| experimento_2_b | 0.999779 |
| experimento_2_c | 0.999821 |

En los tres archivos `x1` y `x2` tienen una correlación casi perfecta (≈ 0.9998): las dos variables contienen prácticamente la misma información. Es una situación de colinealidad casi exacta.

In [3]:
#B
from lista4.models import compare_x1_vs_x1x2

resumen = pd.DataFrame([compare_x1_vs_x1x2(a) for a in archivos]).set_index("archivo")

resumen[["R_x1", "R_x1,x2", "beta1", "beta2"]]

,R_x1,"R_x1,x2",beta1,beta2
archivo,,,,
experimento_2_a,1.029350,1.025907,5.957231,-2.900650
experimento_2_b,1.061539,1.059304,0.606315,2.325776
experimento_2_c,1.141281,1.140041,4.903450,-1.911232


### (b)

| Archivo | R̂ (x1) | R̂ (x1, x2) | β̂₁ | β̂₂ |
|---|---|---|---|---|
| experimento_2_a | 1.0294 | 1.0259 | 5.957 | -2.901 |
| experimento_2_b | 1.0615 | 1.0593 | 0.606 | 2.326 |
| experimento_2_c | 1.1413 | 1.1400 | 4.903 | -1.911 |

- **Riesgos:** en cada archivo, el riesgo con `x1` y el riesgo con `x1` y `x2` son casi iguales. Agregar `x2` apenas reduce el error.
- **Coeficientes:** β̂₁ y β̂₂ del modelo con ambas variables cambian mucho de un archivo a otro, y β̂₂ incluso cambia de signo (negativo en 2a y 2c, positivo en 2b).

In [4]:
#C
tabla = resumen[["corr(x1,x2)", "R_x1", "R_x1,x2", "beta1", "beta2"]]
tabla

,"corr(x1,x2)",R_x1,"R_x1,x2",beta1,beta2
archivo,,,,,
experimento_2_a,0.999760,1.029350,1.025907,5.957231,-2.900650
experimento_2_b,0.999779,1.061539,1.059304,0.606315,2.325776
experimento_2_c,0.999821,1.141281,1.140041,4.903450,-1.911232


In [5]:
#D
resumen["delta_R"] = resumen["R_x1"] - resumen["R_x1,x2"]
resumen[["delta_R"]]

,delta_R
archivo,
experimento_2_a,0.003443
experimento_2_b,0.002235
experimento_2_c,0.001240


### (d) 

ΔR es positivo en los tres archivos, es decir, agregar `x2` siempre reduce el riesgo empírico, pero la reducción es mínima.

In [6]:
#E
def rango(z):
    return z.max() - z.min()

rangos = pd.Series({
    "rango(beta1)": rango(resumen["beta1"]),
    "rango(beta2)": rango(resumen["beta2"]),
    "rango(delta_R)": rango(resumen["delta_R"]),
}, name="valor")

rangos.to_frame()

,valor
rango(beta1),5.350916
rango(beta2),5.226426
rango(delta_R),0.002203


### (e) Rangos 

| Cantidad | Valores (2a, 2b, 2c) | Rango (máx − mín) |
|---|---|---|
| β̂₁ | 5.957, 0.606, 4.903 | 5.3509 |
| β̂₂ | -2.901, 2.326, -1.911 | 5.2264 |
| ΔR | 0.00344, 0.00224, 0.00124 | 0.0022 |

- **rango(β̂₁) = 5.35:** el peso que el modelo le asigna a `x1` cambió en más de 5 unidades entre archivos (de 0.61 a 5.96), un cambio grande.
- **rango(β̂₂) = 5.23:** lo mismo ocurre con el peso de `x2`, que varía más de 5 unidades y llega a cambiar de signo (negativo en 2a y 2c, positivo en 2b).
- **rango(ΔR) = 0.0022:** la reducción del riesgo al agregar `x2` es casi la misma en los tres archivos y es diminuta.

El modelo predice igual de bien en los tres archivos, pero los pesos de `x1` y `x2` cambian por completo. Esto es una consecuencia de la colinealidad entre `x1` y `x2` (correlación ≈ 0.9998): como las dos variables contienen casi la misma información, los datos no permiten determinar cómo repartir el efecto entre ellas.

### (f) 

**¿La incorporación de x2 produce un cambio importante en el riesgo empírico?**

No. En los tres archivos ΔR está entre 0.0012 y 0.0034, lo que equivale a una reducción muy baja. Ademas el riesgo del modelo con `x1` (que ronda 1.03 a 1.14), no cambia mucho al agregar `x2`. Esto se explica porque `x2` es casi una copia de `x1` (correlación ≈ 0.9998): no aporta información nueva para predecir `y`.

**¿Los coeficientes del modelo con ambas variables cambian en la misma proporción entre los tres archivos?**

No. Mientras el riesgo y ΔR casi no se mueven, los coeficientes cambian mucho y de forma desigual: β̂₁ pasa de 5.96 a 0.61 y a 4.90 (rango ≈ 5.35), y β̂₂ pasa de -2.90 a 2.33 y a -1.91 (rango ≈ 5.23), incluso con cambio de signo. Los rangos de los coeficientes son unas 2400 veces mayores que el de ΔR.

**Conclusión:** con variables casi colineales hay muchas combinaciones (β̂₁, β̂₂) que predicen casi igual de bien, así que cada muestra elige una distinta. Por eso el riesgo es estable pero los coeficientes individuales no lo son. Lo que sí es estable es su suma, β̂₁ + β̂₂ ≈ 3 (3.06, 2.93 y 2.99): como `x2` es casi igual a `x1`, el modelo se comporta como uno de una sola variable con pendiente β̂₁ + β̂₂, que es el efecto conjunto de ambas variables.